# 01 Indexing and Lookup

This notebook audits how users find things inside a `Trace`. TorchLens supports Python-style ordinal lookup, structured layer labels, module-call lookup by module path, and accessors for layers, ops, modules, parameters, and buffers.

We use named submodules plus a registered buffer so the lookup examples cover both ordinary layers and module paths.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import torch
from torch import nn
import torchlens as tl

torch.manual_seed(11)


class LookupNet(nn.Module):
    """Small model with named modules and a buffer for lookup examples."""

    def __init__(self) -> None:
        """Initialize modules and a broadcast offset buffer."""
        super().__init__()
        self.stem = nn.Linear(4, 5)
        self.act = nn.ReLU()
        self.head = nn.Linear(5, 2)
        self.register_buffer("offset", torch.tensor([0.25, -0.25]))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run one forward pass."""
        h = self.act(self.stem(x))
        return self.head(h) + self.offset


model = LookupNet().eval()
x = torch.randn(2, 4)
trace = tl.trace(model, x)
print(trace.summary(fields=["name", "shape", "params"]))

Integer indexing is zero-based and follows the visible trace order. Negative indices work like normal Python indexing.

In [ ]:
for index in [0, 1, -1]:
    record = trace[index]
    print(
        f"trace[{index!r}] -> {record.layer_label:12s} type={type(record).__name__} shape={record.shape}"
    )

Layer labels are stable identifiers such as `linear_1_1`. A pass-qualified label adds `:1`, which is useful when a layer position runs more than once.

In [ ]:
linear_label = trace[1].layer_label
qualified_label = f"{linear_label}:1"
print(f"base label: {linear_label} -> {trace[linear_label].layer_label}")
print(f"qualified label: {qualified_label} -> {trace[qualified_label].layer_label}")

Substring lookup is convenient when it is unambiguous. When a substring matches multiple records, TorchLens raises a clear error rather than guessing.

In [ ]:
for key in ["relu", "linear"]:
    try:
        record = trace[key]
        print(f"{key!r} resolved to {record.layer_label}")
    except ValueError as exc:
        print(f"{key!r} is ambiguous: {exc}")

Module paths such as `stem` and `head` resolve to module-call records. A `ModuleCall` is one invocation of a PyTorch module during this trace.

In [ ]:
for module_path in ["stem", "act", "head"]:
    call = trace[module_path]
    print(f"{module_path:>4s} -> {type(call).__name__} {call.call_label}")
    print(f"      inputs={call.input_layers} outputs={call.output_layers} ops={list(call.ops)}")

The accessor properties are the best way to scan groups of records. They are iterable and also support targeted lookup.

In [ ]:
print(f"layers:  {len(trace.layers)} -> {[layer.layer_label for layer in trace.layers]}")
print(f"ops:     {len(trace.ops)} -> {[op.layer_label for op in trace.ops]}")
print(f"modules: {len(trace.modules)} -> {[module.address for module in trace.modules]}")
print(f"params:  {len(trace.params)} -> {[param.address for param in trace.params]}")
print(f"buffers: {len(trace.buffers)} -> {[buffer.address for buffer in trace.buffers]}")

You can move from a module to its calls and outputs, then back to the corresponding layers.

In [ ]:
head_module = trace.modules["head"]
print(f"module address: {head_module.address}")
call_labels = [head_module.calls[position].call_label for position in range(len(head_module.calls))]
print(f"call labels: {call_labels}")
for layer_label in head_module.output_layers:
    layer = trace[layer_label]
    print(f"output layer {layer.layer_label}: shape={layer.shape} parents={layer.parents}")

## 🔧 Sandbox

Try changing `lookup_key` to an integer, a full layer label, a substring like `add`, or a module path like `stem`.

In [ ]:
# TODO: try lookup_key = 2, -1, "add", "stem", or trace.output_layers[0].
lookup_key = "head"
record = trace[lookup_key]
print(f"lookup {lookup_key!r} -> {type(record).__name__}")
print(record)